# KC aggregation exploration

This notebook investigates how to reduce the granularity of Knowledge Components (KCs) before fitting a BKT model. Using the KC prerequisite graph, it compares three aggregation strategies: chain collapsing (grouping each KC under its top-level ancestor), community detection (grouping KCs by graph modularity clusters) and manual grouping. Each approach is evaluated by examining the resulting observation distribution per coarse KC and fitting a pyBKT model on the aggregated data.

## Imports and Data

In [1]:
import pandas as pd
import networkx as nx
from pyBKT.models import Model
from sklearn.model_selection import train_test_split

In [2]:
nodes = pd.read_excel('../data/raw/Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx', sheet_name='KC_Nodes')
edges = pd.read_excel('../data/raw/Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx', sheet_name='KC_Edges')
obs = pd.read_excel('../data/raw/Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx', sheet_name='Student_Observations')
group = pd.read_excel('../data/raw/mkc_mapping_pack_v1.0..xlsx', sheet_name=1)

The data comes from three sheets in `final_data.xlsx`:

- **`KC_Nodes`** — one row per Knowledge Component (KC), with a unique `kc_id`
- **`KC_Edges`** — prerequisite relationships between KCs (`source_kc_id` → `target_kc_id`), defining the directed graph
- **`Student_Observations`** — one row per student attempt, with `student_id`, `primary_kc_id`, `observation_id`, and `score`

A fourth file (`mkc_mapping_pack_v1.0.xlsx`) contains a hand-crafted mapping from fine-grained KCs to coarser *modelling KCs*, used in Approach 3.

## Approach 1 : Collapse chains

First, we go up the knowledge graph and assign each KC to its predecessors until we reach the root. This root KC is the one we use for modelling.

In [3]:
G = nx.DiGraph()
G.add_nodes_from(nodes["kc_id"])

# Only use prerequisite edges to define hierarchy
G.add_edges_from(zip(edges["source_kc_id"], edges["target_kc_id"]))

# Find root nodes (no predecessors = top of the chain)
roots = [n for n in G.nodes if G.in_degree(n) == 0]

# Assign each KC to its earliest ancestor (root)
kc_to_group = {}
for root in roots:
    descendants = nx.descendants(G, root) | {root}
    for kc in descendants:
        kc_to_group[kc] = root  # or overwrite with most recent ancestor

nodes["app_one"] = nodes["kc_id"].map(kc_to_group)

## Approach 2 :

Group KCs into clusters, where every KC that is related is in the same cluster.

### How it works

The KC prerequisite graph is converted to an **undirected graph** and then partitioned using the [greedy modularity community detection](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.modularity_max.greedy_modularity_communities.html) algorithm from NetworkX. KCs that are densely connected to each other end up in the same cluster, regardless of where they sit in the hierarchy.

Unlike Approach 1, this method does not require a strict tree structure — it can handle KCs with multiple prerequisites and will group KCs across different branches if they are sufficiently interconnected.

In [4]:
import networkx.algorithms.community as nx_comm

G_undirected = G.to_undirected()
communities = nx_comm.greedy_modularity_communities(G_undirected)

kc_to_community = {}
for i, community in enumerate(communities):
    for kc in community:
        kc_to_community[kc] = f"cluster_{i}"

nodes["app_two"] = nodes["kc_id"].map(kc_to_community)

In [5]:
clusters = nodes[['kc_id','app_two']].groupby('app_two').agg('sum')
clusters

,kc_id
app_two,
cluster_0,KC.U5.05.sample_size_effect_variabilityKC.U5.1...
cluster_1,KC.U6.07.one_prop_ci_interpretKC.U6.09.ci_clai...
cluster_10,KC.U8.01.categorical_count_data_structureKC.U8...
cluster_11,KC.U7.18.two_sample_t_interval_procedure_selec...
cluster_12,KC.U10.01.bootstrap_purpose_sampling_distribut...
cluster_13,KC.U7.02.t_distribution_degrees_freedomKC.U7.0...
cluster_14,KC.U9.12.slope_interval_width_sample_size
cluster_2,KC.U5.09.sample_proportion_meanKC.U5.10.sample...
cluster_3,KC.U4.01.probability_long_run_interpretationKC...


The table above shows the 15 clusters produced by community detection. Each row concatenates the `kc_id` strings of all KCs in that cluster. The clusters vary in size: most contain KCs from a single statistical unit (e.g. sampling distributions), while a few bridge related units.

## Approach 3 : 

Use the manual grouping given in the `group` data.

### How it works

The mapping file stores each modelling KC (`modeling_kc_id`) alongside a semicolon-separated list of fine-grained KCs (`fine_kc_ids`) that map to it. The code explodes this list so each fine KC gets its own row, then left-joins onto the nodes table to assign every KC a `modeling_kc_id` (stored as `app_three`). KCs not covered by the mapping will have a null value for `app_three`.

In [6]:
group['kc_id'] = group['fine_kc_ids'].str.split('; ')
group = group.explode('kc_id')   
nodes = pd.merge(nodes, 
         group[['modeling_kc_id','kc_id']],
         on='kc_id',
         how='left').rename(columns={'modeling_kc_id':'app_three'})    

## Observation distributions

Comparing the number of groups for each approach, we have :

In [7]:
groups = pd.DataFrame({'Approach':['Baseline','Approach 1', 'Approach 2', 'Approach 3'],
                'Number of groups' : [nodes['kc_id'].nunique(),nodes['app_one'].nunique(),nodes['app_two'].nunique(),nodes['app_three'].nunique()]}).sort_values('Number of groups')

groups

,Approach,Number of groups
2,Approach 2,15
1,Approach 1,41
3,Approach 3,47
0,Baseline,256


| Approach | Groups | Reduction from baseline |
|---|---|---|
| Baseline (`primary_kc_id`) | 256 | — |
| Approach 1 (chain collapse) | 41 | 84% |
| Approach 3 (manual) | 47 | 82% |
| Approach 2 (community detection) | 15 | 94% |

Approach 2 produces the fewest groups by far. Approaches 1 and 3 are similar in granularity, though they partition the KCs differently — chain collapsing is purely graph-driven, while the manual mapping reflects subject-matter judgement.

Comparing the number of observations across distributions :

In [8]:
import altair as alt
for approach in ['primary_kc_id','app_one', 'app_two', 'app_three'] :
    if approach != 'primary_kc_id' :
            #  Create a simple lookup: fine KC -> coarse KC
            kc_mapping = nodes[["kc_id", approach]].set_index("kc_id")[approach].to_dict()
        
            # Map primary_kc_id to its coarse KC
            obs[approach] = obs["primary_kc_id"].map(kc_mapping)
    
    # Count observations per coarse KC
    obs_counts = obs.groupby(approach).agg(
        n_observations=("observation_id", "count"),
        n_students=("student_id", "nunique"),
    ).reset_index().sort_values('n_observations')

    obs_counts['n_obs_per_stu'] = obs_counts['n_observations'] / obs_counts['n_students']
    
    chart = alt.Chart(obs_counts).mark_bar().encode(
        x=alt.X('n_observations', title = 'Number of Observations').bin(),
        y='count()',
    ).properties(title=f'Observation distribution — {approach}')
    chart.display()


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

**Key observations from the histograms:**

- The **baseline** distribution is right-skewed (23–424 observations per KC), with most KCs having fewer than 200 observations — potentially too sparse for reliable BKT parameter estimation.
- **Approach 1** (chain collapse) has a heavily right-skewed distribution: a few root KCs accumulate thousands of observations while many still have very few, suggesting some chains are much deeper than others.
- **Approach 2** (community detection) shows a bimodal pattern — a few very small clusters and many large ones — reflecting how modularity optimisation tends to create a mix of tight small communities and larger loosely-connected ones.
- **Approach 3** (manual) has the most even spread, which is expected since the mapping was designed by subject experts with sensible groupings in mind.

## pyBKT

Testing out pyBKT with the new dataset :

### Data preprocessing for pyBKT

The `preprocess` function prepares a dataframe for pyBKT by:

1. **Binarising scores** — partial credit (0.5) is treated as incorrect (0), so `correct ∈ {0, 1}`
2. **Renaming columns** to match pyBKT's expected schema: `user_id`, `skill_name`, `correct`, `order_id`
3. **Recomputing `order_id`** as a per-student cumulative count, ensuring observations are ordered correctly within each student's sequence

The `kc_col` parameter controls which aggregation column is used as the skill label, allowing the same function to be reused across all four approaches.

In [9]:
def preprocess(data, kc_col='primary_kc_id'):
  # Consider any non-full marks as incorrect (replace 0.5 -> 0)
  data['correct'] = data['score'].case_when([
      (data['score'] == 0.5, 0)
  ]).astype(int)

  # Rename  columns
  obs = data.rename(columns={
      'student_id':     'user_id',
      kc_col:  'skill_name',
      'observation_id': 'order_id'
  })[['user_id', 'skill_name', 'correct', 'order_id']].reset_index(drop=True)
  obs['order_id'] = obs.groupby('user_id').cumcount()
  return obs



Each approach is evaluated with a **70/30 student-level train/test split** (stratified by student, not by observation, to avoid data leakage). A separate BKT model is fit per approach using `num_fits=1` for speed. Two metrics are reported:


In [10]:
eval_metrics = {'approach':[], 'RMSE':[], 'AUC':[]}

for approach in ['primary_kc_id','app_one', 'app_two', 'app_three'] :
    if approach != 'primary_kc_id' :
        #  Create a simple lookup: fine KC -> coarse KC
        kc_mapping = nodes[["kc_id", approach]].set_index("kc_id")[approach].to_dict()
    
        # Map primary_kc_id to its coarse KC
        obs[approach] = obs["primary_kc_id"].map(kc_mapping)
    student_id = obs.groupby('student_id').agg('count').reset_index()[['student_id']]
    train_data, test_data = train_test_split(student_id, test_size=0.3, random_state=42)
    train_data = pd.merge(train_data.sort_values('student_id'),obs, on='student_id', how='left')
    test_data = pd.merge(test_data.sort_values('student_id'),obs, on='student_id', how='left')

    # Preprocess train and test data
    proc_train = preprocess(data=train_data, kc_col=approach)
    proc_test = preprocess(test_data, kc_col=approach)

    model = Model(seed=42, num_fits=1) 
    model.fit(data = proc_train)

    eval_metrics['approach'].append(approach)
    eval_metrics['RMSE'].append(model.evaluate(data = proc_test))
    eval_metrics['AUC'].append(model.evaluate(data = proc_test, metric = 'auc'))


- **RMSE** — root mean squared error on predicted correctness probabilities (lower is better)
- **AUC** — area under the ROC curve (higher is better; 0.5 = random, 1.0 = perfect)

In [11]:
results = pd.DataFrame(eval_metrics)
results

,approach,RMSE,AUC
0,primary_kc_id,0.50162,0.58517
1,app_one,0.48769,0.63580
2,app_two,0.48821,0.63107
3,app_three,0.48980,0.62630


In [12]:
# Save results to csv


merged = groups.merge(results, left_index=True, right_index=True)
merged = merged.sort_index().reset_index(drop=True)
merged = merged.drop(columns=['approach']).rename(columns={'Approach': 'approach'})

merged.to_csv('../data/outputs/kc_agg_results.csv', index=False)

### Results interpretation

All three aggregation approaches outperform the baseline on both metrics, confirming that the fine-grained KCs are too sparse for BKT to estimate reliable parameters. Aggregating KCs before fitting consistently improves both predictive accuracy (lower RMSE) and discriminative ability (higher AUC).

Approach 2 (community detection) achieves the best scores overall (RMSE: 0.485, AUC: 0.643), followed closely by Approach 1 (chain collapse) and Approach 3 (manual grouping).

### Metrics vs. interpretability tradeoff

While Approach 2 leads on raw metrics, it produces only 15 clusters with a heavily skewed observation distribution — a few clusters accumulate thousands of observations while others remain sparse. The cluster labels (`cluster_0`, `cluster_1`, ...) also carry no semantic meaning, making it harder to interpret BKT parameter estimates or communicate results.

Approach 3 (manual grouping) is only marginally worse (AUC: 0.626 vs 0.643) but produces semantically meaningful KC groups and a more even observation distribution. Depending on whether the goal is prediction performance or interpretable skill modelling, Approach 3 may be the more practical choice.

### Caveats

These results should be treated as preliminary given several limitations of the evaluation setup:

- **Small sample**: only 25 students total, meaning the 70/30 split yields ~7 test students. A single student with unusual response patterns could noticeably shift the AUC.
- **Single fit**: `num_fits=1` was used for speed. BKT fits can be sensitive to parameter initialisation, so the reported values may not reflect the best achievable performance for each approach.
- **Single train/test split**: with a fixed random seed and no cross-validation, there is no estimate of variance around these metrics. The differences between approaches (< 0.02 RMSE, < 0.06 AUC) may not be meaningful at this scale.